In [5]:
# pip install pulp  # once
import os, re
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd
import pulp

CLASSES = ["High","Good","Moderate","Bad","Poor"]
C = len(CLASSES); CLS2ID = {c:i for i,c in enumerate(CLASSES)}

def tidy_label(x):
    if x is None or (isinstance(x,float) and pd.isna(x)): return None
    s = str(x).strip().lower().replace('_',' ').replace('-',' ')
    s = s.title()
    return {"High":"High","Good":"Good","Moderate":"Moderate","Bad":"Bad","Poor":"Poor"}.get(s, None)

def parse_acc_from_name(name: str) -> float:
    m = list(re.finditer(r'acc\s*=\s*([0-9]+(?:\.[0-9]+)?)\s*%?', name, flags=re.I))
    if not m: raise ValueError(f"Missing acc=... in: {name}")
    s = m[-1].group(1); v = float(s)
    has_pct = '%' in name[m[-1].start(): m[-1].end()]
    if has_pct or v > 1.0: v /= 100.0
    return float(np.clip(v, 1e-3, 1-1e-3))

def load_votes(pred_dir: Path) -> Tuple[pd.Index, List[str], np.ndarray, List[pd.Series]]:
    paths = sorted([p for p in pred_dir.iterdir() if p.suffix.lower() in (".csv",".tsv",".parquet",".feather")])
    if not paths: raise RuntimeError(f"No files in {pred_dir}")
    file_names, accs, series = [], [], []
    for p in paths:
        if p.suffix.lower()==".csv":      df = pd.read_csv(p)
        elif p.suffix.lower()==".tsv":    df = pd.read_csv(p, sep="\t")
        elif p.suffix.lower()==".parquet":df = pd.read_parquet(p)
        elif p.suffix.lower()==".feather":df = pd.read_feather(p)
        else: continue

        if "SamplingOperations_code" in df.columns:
            df = df.set_index(df["SamplingOperations_code"].astype(str)).drop(columns=["SamplingOperations_code"])
        else:
            if df.index.name != "SamplingOperations_code":
                raise ValueError(f"{p.name} lacks SamplingOperations_code")
            df.index = df.index.astype(str)

        # detect prediction column
        cand = [c for c in df.columns if c not in ("SamplingOperations_code",)]
        col = cand[0] if len(cand)==1 else next((c for c in
                ["IBD_EQR_Status","predictions","label","class","y_hat","y"] if c in df.columns), None)
        if col is None: raise ValueError(f"Cannot infer label column in {p.name}")

        s = df[col].map(tidy_label)
        if s.isna().all(): raise ValueError(f"All labels NA in {p.name}")
        series.append(s)
        file_names.append(p.name)
        accs.append(parse_acc_from_name(p.name))

    items = pd.Index(sorted(set().union(*[set(s.index) for s in series])), name="SamplingOperations_code")
    return items, file_names, np.array(accs, dtype=float), series


In [6]:
a=1

In [7]:
def solve_consensus_anchor(items: pd.Index,
                           file_names: List[str],
                           accs: np.ndarray,
                           series: List[pd.Series],
                           anchor: str | None = None,
                           round_mode: str = "nearest",
                           time_limit: int | None = None):
    """
    Phase 1: minimize total slack so per-model agreement ≈ reported accuracy.
    Phase 2: fix slack at optimum and maximize agreement with 'anchor' model.
    Returns: y_hat (list[str]), agreement table (DataFrame), slack summary (DataFrame).
    """
    N = len(items); K = len(file_names)
    # Build per-model coverage and target hit counts
    Sk_rows, Lk_ids, nk, tk = [], [], [], []
    for k, s in enumerate(series):
        s_al = s.reindex(items)
        idx = s_al.dropna().index
        rows = [items.get_loc(ix) for ix in idx]
        labs = s_al.loc[idx].map(lambda z: CLS2ID[z]).astype(int).values
        Sk_rows.append(rows)
        Lk_ids.append(labs)
        nk.append(len(rows))
        target = accs[k] * len(rows)
        if round_mode == "nearest": t = int(np.rint(target))
        elif round_mode == "floor": t = int(np.floor(target))
        elif round_mode == "ceil":  t = int(np.ceil(target))
        else: raise ValueError("round_mode ∈ {'nearest','floor','ceil'}")
        tk.append(t)
    nk = np.array(nk, int); tk = np.array(tk, int)

    # Select anchor
    if anchor is None:
        anch_id = int(np.nanargmax(accs))
    else:
        anch_id = file_names.index(anchor)

    # ---------- Phase 1: minimize slack ----------
    prob1 = pulp.LpProblem("ConsensusMinSlack", pulp.LpMinimize)

    # x[i,c] ∈ {0,1}
    X = {(i,c): pulp.LpVariable(f"x_{i}_{c}", lowBound=0, upBound=1, cat="Binary")
         for i in range(N) for c in range(C)}

    # one class per item
    for i in range(N):
        prob1 += pulp.lpSum(X[(i,c)] for c in range(C)) == 1, f"one_{i}"

    # per-model agreement with integer slack
    sp, sm = [], []
    for k in range(K):
        sp_k = pulp.LpVariable(f"s_plus_{k}", lowBound=0, cat="Integer")
        sm_k = pulp.LpVariable(f"s_minus_{k}", lowBound=0, cat="Integer")
        sp.append(sp_k); sm.append(sm_k)
        if nk[k] == 0:   # no coverage
            prob1 += sp_k == 0; prob1 += sm_k == 0
            continue
        agree = pulp.lpSum(X[(Sk_rows[k][j], Lk_ids[k][j])] for j in range(nk[k]))
        prob1 += agree + sm_k - sp_k == tk[k], f"acc_{k}"

    # objective: minimize total slack
    prob1 += pulp.lpSum(sp) + pulp.lpSum(sm)

    solver1 = pulp.PULP_CBC_CMD(msg=True, timeLimit=time_limit) if time_limit else pulp.PULP_CBC_CMD(msg=True)
    st1 = prob1.solve(solver1)
    stat1 = pulp.LpStatus[st1]
    if stat1 not in ("Optimal","Feasible"):
        raise RuntimeError(f"Phase 1 solver status: {stat1}")

    # read optimal slack
    sp_star = np.array([int(pulp.value(v)) for v in sp], dtype=int)
    sm_star = np.array([int(pulp.value(v)) for v in sm], dtype=int)
    slack_tbl = pd.DataFrame({
        "modelo": file_names, "n": nk, "target_hits": tk,
        "s_plus": sp_star, "s_minus": sm_star, "net": sm_star - sp_star
    })

    # ---------- Phase 2: fix slack and maximize anchor matches ----------
    prob2 = pulp.LpProblem("ConsensusMaxAnchor", pulp.LpMaximize)

    # reuse X variables for the same names to warm start
    X2 = {(i,c): pulp.LpVariable(f"x_{i}_{c}", lowBound=0, upBound=1, cat="Binary")
          for i in range(N) for c in range(C)}

    # one class per item
    for i in range(N):
        prob2 += pulp.lpSum(X2[(i,c)] for c in range(C)) == 1, f"one2_{i}"

    # fix slack to phase-1 values and re-impose equality
    for k in range(K):
        if nk[k] == 0: 
            continue
        agree = pulp.lpSum(X2[(Sk_rows[k][j], Lk_ids[k][j])] for j in range(nk[k]))
        # agree == tk[k] - sm_star[k] + sp_star[k]
        prob2 += agree == int(tk[k] - sm_star[k] + sp_star[k]), f"acc_fix_{k}"

    # anchor objective: maximize overlap with anchor labels on its covered rows
    rows_a, labs_a = Sk_rows[anch_id], Lk_ids[anch_id]
    obj = pulp.lpSum(X2[(rows_a[j], labs_a[j])] for j in range(len(rows_a)))
    prob2 += obj

    solver2 = pulp.PULP_CBC_CMD(msg=True, timeLimit=time_limit) if time_limit else pulp.PULP_CBC_CMD(msg=True)
    st2 = prob2.solve(solver2)
    stat2 = pulp.LpStatus[st2]
    if stat2 not in ("Optimal","Feasible"):
        raise RuntimeError(f"Phase 2 solver status: {stat2}")

    # extract y_hat
    xmat = np.zeros((N, C), dtype=int)
    for i in range(N):
        for c in range(C):
            xmat[i,c] = int(pulp.value(X2[(i,c)]) > 0.5)
    y_idx = xmat.argmax(axis=1)
    y_hat = [CLASSES[j] for j in y_idx]

    # verify agreements
    check = []
    for k in range(K):
        if nk[k] == 0:
            coinc = np.nan
        else:
            rows = Sk_rows[k]
            labs = Lk_ids[k]
            coinc = (y_idx[rows] == labs).mean()
        check.append({"modelo": file_names[k],
                      "accuracy_reportada": accs[k],
                      "coincidencia_con_df": coinc})
    check_df = pd.DataFrame(check)
    return y_hat, y_idx, check_df, slack_tbl, file_names[anch_id]


# integer programmitng

In [8]:
# Directory with your files
dir_win   = r"results\cleaned"
dir_posix = "results/cleaned"
pred_dir = Path(dir_win) if Path(dir_win).exists() else Path(dir_posix)

items, file_names, accs, series = load_votes(pred_dir)

# Choose anchor automatically (highest reported accuracy) or set anchor="PredictionX_acc=....csv"
anchor = None

y_hat, y_idx, check_df, slack_tbl, anchor_name = solve_consensus_anchor(
    items, file_names, accs, series,
    anchor=anchor,
    round_mode="nearest",
    time_limit=None
)

# Save results
out = pd.DataFrame({"y_hat": y_hat}, index=items)
pred_dir = Path("results/yhat construction/yolo")
out.to_csv(pred_dir / "Yhat_IntegerProgramming.csv")
check_df.to_csv(pred_dir / "agreement_check.csv", index=False)
slack_tbl.to_csv(pred_dir / "slack_summary.csv", index=False)

print("Anchor:", anchor_name)
display(check_df)
display(slack_tbl[(slack_tbl.s_plus!=0)|(slack_tbl.s_minus!=0)])
print("Saved:", pred_dir / "y_hat_consensus_anchor.csv")


Anchor: CBFullTraining_acc=0.899664.csv


,modelo,accuracy_reportada,coincidencia_con_df
0,BC_acc=0.820462.csv,0.820462,0.820462
1,CBCr_acc=0.833695.csv,0.833695,0.833695
2,CBFullTraining_acc=0.899664.csv,0.899664,0.899644
3,CBph_acc=0.852459.csv,0.852459,0.852459
4,CBt_acc=0.8689.csv,0.868900,0.868852
5,Interpolation_acc=0.311300.csv,0.311300,0.311278
6,JAPredictionsXGB_acc=0.816907.csv,0.816907,0.816907
7,RandomPredictions_acc=0.3075.csv,0.307500,0.307525
8,RFG_acc=0.814100.csv,0.814100,0.814142
9,RFpR_acc=0.802489.csv,0.802489,0.802489


,modelo,n,target_hits,s_plus,s_minus,net


Saved: results\yhat construction\yolo\y_hat_consensus_anchor.csv
